<a href="https://colab.research.google.com/github/zkhadija132/flyrank-ml-assignment/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I selected the Random Forest Classifier because it performs well on structured tabular data and can capture non-linear relationships between features. It is more robust than a simple rule-based baseline and also provides feature importance for interpretation.

In [9]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

print(model)

RandomForestClassifier(random_state=42)


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

The dataset was split into 80% training data and 20% testing data using train_test_split with random_state=42. This split provides enough data for training while keeping a separate test set for evaluation.

In [2]:
from huggingface_hub import login
login()

In [3]:
from datasets import load_dataset
import pandas as pd

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    streaming=True,
    split="train"
)

df = pd.DataFrame(ds.take(1000))

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

In [4]:
df["baseline_score"] = (
    df["gsc_clicks"].fillna(0) * 0.4 +
    df["ga4_sessions"].fillna(0) * 0.3 +
    df["scroll_events"].fillna(0) * 0.2 +
    df["ga4_engaged_sessions"].fillna(0) * 0.1
)

In [5]:
X = df[["gsc_clicks", "ga4_sessions", "scroll_events"]]
y = (df["baseline_score"] > df["baseline_score"].median()).astype(int)

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training data shape:", X_train.shape)
print("Testing data shape:", X_test.shape)

Training data shape: (800, 3)
Testing data shape: (200, 3)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

The Random Forest model was trained using the same features as the baseline. The model performance was evaluated on the test data and compared with the Week 4 baseline. The trained model provides more flexible decision boundaries than the rule-based baseline.

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Model Accuracy:", accuracy_score(y_test, pred))
print()
print(classification_report(y_test, pred))

Model Accuracy: 1.0

              precision    recall  f1-score   support

           0       1.00      1.00      1.00       188
           1       1.00      1.00      1.00        12

    accuracy                           1.00       200
   macro avg       1.00      1.00      1.00       200
weighted avg       1.00      1.00      1.00       200



## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The model correctly identified most high-activity content but may misclassify low-activity items because the available engagement signals are limited. GSC clicks contributed the most to the predictions, while GA4 sessions had little impact because almost all values were zero in the sample.

In [8]:
importance = model.feature_importances_

for feature, score in zip(X.columns, importance):
    print(feature, ":", round(score, 4))

gsc_clicks : 1.0
ga4_sessions : 0.0
scroll_events : 0.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.